In [73]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import mean_squared_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout, BatchNormalization
import warnings


warnings.filterwarnings('ignore')
tf.get_logger().setLevel('ERROR')

def preprocess(df):
    df = df.dropna(subset=["price", "rrp", "sizeCode"])
    df = df[(df["rrp"] > 0) & (df["price"] > 0) & (df["price"] <= df["rrp"] * 1.3)]
    df['orderDate'] = pd.to_datetime(df['orderDate'])
    df['order_month'] = df['orderDate'].dt.month
    df['order_week'] = df['orderDate'].dt.dayofweek
    df['order_year'] = df['orderDate'].dt.year
    df = df.drop(['orderID', 'orderDate', 'voucherID', 'deviceID', 'customerID', 'articleID'], axis=1)
    return df

test = pd.read_csv("..\\..\\Задание 1. Классические задачи машинного обучения\\1) Регрессия\\train.csv")
train = pd.read_csv("..\\..\\Задание 1. Классические задачи машинного обучения\\1) Регрессия\\test.csv")

train_processed = preprocess(train)
test_processed = preprocess(test)

X_train = train_processed.drop('price', axis=1)
y_train = train_processed['price']

X_test = test_processed.drop('price', axis=1)
y_test = test_processed['price']

numeric = ['voucherAmount', 'rrp', 'order_month', 'order_week', 'order_year', "quantity"]
categorical = ['colorCode', 'productGroup', 'paymentMethod']

encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
train_cat_encoded = encoder.fit_transform(X_train[categorical])#преобразуем тренеровочные данные
test_cat_encoded = encoder.transform(X_test[categorical]) # преобразуем тестовые данные

train_df = pd.DataFrame(train_cat_encoded,
                            index=X_train.index,
                            columns=encoder.get_feature_names_out(categorical))
test_df = pd.DataFrame(test_cat_encoded,
                           index=X_test.index,
                           columns=encoder.get_feature_names_out(categorical))

scaler = StandardScaler()
train_num_scaled = scaler.fit_transform(X_train[numeric])
test_num_scaled = scaler.transform(X_test[numeric])

train_num_df = pd.DataFrame(train_num_scaled,
                            index=X_train.index,
                            columns=numeric)
test_num_df = pd.DataFrame(test_num_scaled,
                           index=X_test.index,
                           columns=numeric)

X_train_final = pd.concat([train_num_df, train_df], axis=1)
X_test_final = pd.concat([test_num_df, test_df], axis=1)

X_test_final = X_test_final[X_train_final.columns]

model = Sequential([
    Input(shape=(X_train_final.shape[1],)),
    Dense(256, activation='relu'),
    BatchNormalization(),
    Dropout(0.4),
    Dense(128, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dense(1)
])
model.compile(optimizer='adam', loss='mse')

model.fit(X_train_final, y_train,
          epochs=50,
          batch_size=256,
          verbose=1)

y_pred = model.predict(X_test_final).flatten()
final_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"  ИТОГОВЫЙ RMSE: {final_rmse:.4f}")

Epoch 1/50
74/74 [==============================] - 1s 4ms/step - loss: 851.2013
Epoch 2/50
74/74 [==============================] - 0s 4ms/step - loss: 225.8216
Epoch 3/50
74/74 [==============================] - 0s 4ms/step - loss: 153.3917
Epoch 4/50
74/74 [==============================] - 0s 4ms/step - loss: 130.7575
Epoch 5/50
74/74 [==============================] - 0s 4ms/step - loss: 119.7509
Epoch 6/50
74/74 [==============================] - 0s 4ms/step - loss: 114.0453
Epoch 7/50
74/74 [==============================] - 0s 4ms/step - loss: 104.8086
Epoch 8/50
74/74 [==============================] - 0s 4ms/step - loss: 102.1184
Epoch 9/50
74/74 [==============================] - 0s 4ms/step - loss: 96.8431
Epoch 10/50
74/74 [==============================] - 0s 4ms/step - loss: 94.0175
Epoch 11/50
74/74 [==============================] - 0s 4ms/step - loss: 90.9457
Epoch 12/50
74/74 [==============================] - 0s 4ms/step - loss: 89.8126
Epoch 13/50
74/74 [==========